In [1]:
import numpy as np
import json
import copy
from typing import List, Tuple, Dict, Any, Set


In [2]:
class PeerToPeerConsensusAHP:
    def __init__(self, threshold: float = 1.03, max_iterations: int = 10):
        self.threshold = threshold
        self.max_iterations = max_iterations
        self.current_pcms = None
        self.dm_ids = None
        self.alternatives = None
        
    def compatibility_index(self, A: np.ndarray, B: np.ndarray) -> float:
        """
        Индекс совместимости между двумя матрицами
        """
        n = A.shape[0]
        hadamard_product = A*B.T
        return (1 / n**2) * np.sum(hadamard_product)
    
    def calculate_ici_matrix(self, all_pcms: List[np.ndarray], D: Set[Tuple[int, int]]) -> np.ndarray:
        """
        Матрица консенсуса - формула 6
        """
        m = len(all_pcms)
        ici_matrix = np.ones((m, m))
        
        for i in range(m):
            for j in range(m):
                if i == j:
                    ici_matrix[i, j] = 1.0
                elif (i, j) in D:
                    ici = self.compatibility_index(all_pcms[i], all_pcms[j])
                    ici_matrix[i, j] = ici
                else:
                    ici_matrix[i, j] = np.nan
                    
        return ici_matrix
    
    def find_max_ici_pair(self, ici_matrix: np.ndarray, D: Set[Tuple[int, int]]) -> Tuple[int, int]:
        """
        Нахождение пары с максимальным ICI из доступных пар
        """
        max_ici = -1
        best_pair = None
        
        for pair in D:
            i, j = pair
            if not np.isnan(ici_matrix[i, j]) and ici_matrix[i, j] > max_ici:
                max_ici = ici_matrix[i, j]
                best_pair = pair
        if best_pair is not None:
            best_pair = tuple(sorted(best_pair))        
        return best_pair
    
    def calculate_alpha(self, ici_matrix: np.ndarray, pair: Tuple[int, int], m: int) -> Tuple[float, float]:
        """
        Расчет коэффициентов по формулам 11-12
        """
        a, b = pair
        
        sum_ici_a = 0
        sum_ici_b = 0
        count = 0
        
        for l in range(m):
            if l != a and l != b and not np.isnan(ici_matrix[a, l]) and not np.isnan(ici_matrix[b, l]):
                sum_ici_a += ici_matrix[a, l]
                sum_ici_b += ici_matrix[b, l]
                count += 1
        
        if count == 0 or (sum_ici_a + sum_ici_b) == 0:
            return 0.5, 0.5
            
        total = sum_ici_a + sum_ici_b
        alpha_a = 1 - (sum_ici_a / (2 * total))
        alpha_b = 1 - (sum_ici_b / (2 * total))
        
        return round(alpha_a,4), round(alpha_b,4)
    
    def update_pcm(self, A_own: np.ndarray, A_other: np.ndarray, alpha: float) -> np.ndarray:
        """
        Обновление матрицы по формулам 9-10
        """
        return np.round(np.power(A_own, alpha) * np.power(A_other, 1 - alpha),3)
    
    def calculate_decision_maker_weights(self, ici_matrix: np.ndarray, m: int) -> np.ndarray:
        """
        Расчет весов экспертов через марковскую цепь - 7-8
        """
        P = np.zeros((m, m))
        
        for i in range(m):
            denominators = []
            for j in range(m):
                if i != j and not np.isnan(ici_matrix[i, j]):
                    denominators.append(1 / ici_matrix[i, j])
            
            if denominators:
                denominator_sum = sum(denominators)
                for j in range(m):
                    if i != j and not np.isnan(ici_matrix[i, j]):
                        P[i, j] = (1 / ici_matrix[i, j]) / denominator_sum
        n = P.shape[0]
        A = P.T - np.eye(n)
        A = np.vstack([A, np.ones(n)])
        b = np.zeros(n + 1)
        b[-1] = 1
        
        pi, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        weights = np.abs(pi) / np.abs(pi).sum()
        
        return weights
    
    def aggregate_group_pcm(self, all_pcms: List[np.ndarray], weights: np.ndarray) -> np.ndarray:
        n = all_pcms[0].shape[0]
        log_G = np.zeros((n, n))
        
        for k, A_k in enumerate(all_pcms):
            log_G += weights[k] * np.log(A_k)
        
        return np.exp(log_G)
    
    def calculate_priority_vector(self, A: np.ndarray) -> np.ndarray:
        n = A.shape[0]
        geometric_means = np.prod(A, axis=1) ** (1/n)
        return np.round(geometric_means / geometric_means.sum(),3)
    
    def check_consistency(self, A: np.ndarray) -> Tuple[float, float]:
        n = A.shape[0]
        
        RI_table = {1: 0, 2: 0, 3: 0.52, 4: 0.89, 5: 1.11, 
                   6: 1.25, 7: 1.35, 8: 1.40, 9: 1.45, 10: 1.49}
        RI = RI_table.get(n, 1.49)
        
        eigenvalues, _ = np.linalg.eig(A)
        lambda_max = max(eigenvalues.real)
        
        CI = (lambda_max - n) / (n - 1) if n > 1 else 0
        CR = CI / RI if RI > 0 else 0
        
        return CI, CR

In [3]:
def enhanced_interactive_acceptance_decision(self, pair: Tuple[int, int], iteration: int, 
                                           dm_ids: List[str], ici_matrix: np.ndarray, D: Set[Tuple[int, int]] ) -> Tuple[bool, bool]:
    a, b = pair
    ici_value = ici_matrix[a, b]
    
    print(f"\n{'='*80}")
    print(f"ИТЕРАЦИЯ {iteration + 1}")
    print(f"{'='*80}")
    
    # Показ матрицы ICI
    print(f"\nМАТРИЦА ИНДИВИДУАЛЬНЫХ ИНДЕКСОВ КОНСЕНСУСА (ICI):")
    self.print_ici_matrix(ici_matrix, dm_ids)
    
    alpha_a, alpha_b = self.calculate_alpha(ici_matrix, (a, b), len(dm_ids))
    
    proposed_A = self.update_pcm(self.current_pcms[a], self.current_pcms[b], alpha_a)
    proposed_B = self.update_pcm(self.current_pcms[b], self.current_pcms[a], alpha_b)
    
    print(f"\nПРЕДЛОЖЕНИЕ ПО ПЕРЕСМОТРУ:")
    print(f"   {dm_ids[a]} сохранит {alpha_a} своего мнения")
    print(f"   {dm_ids[b]} сохранит {alpha_b} своего мнения")

    can_a_change = (a, b) in D  # DM_a может изменить?
    can_b_change = (b, a) in D  # DM_b может изменить?

    if can_a_change:
        # Спрашиваем DM_a
        print(f"\nПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ {dm_ids[a]}:")
        self.print_matrix_comparison(self.current_pcms[a], proposed_A, f"Текущая {dm_ids[a]}", f"Предлагаемая {dm_ids[a]}")
        while True:
            response_a = input(f"   {dm_ids[a]} принимает? (y/n): ").strip().lower()
            if response_a in ['y', 'yes', 'да', 'д']:
                accept_a = True
                break
            elif response_a in ['n', 'no', 'нет', 'н']:
                accept_a = False
                break
            else:
                print(" Пожалуйста, введите 'y', 'n'")
    else:
        accept_a = False  # Не спрашиваем
    
    if can_b_change:
        print(f"\nПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ {dm_ids[b]}:")
        self.print_matrix_comparison(self.current_pcms[b], proposed_B, f"Текущая {dm_ids[b]}", f"Предлагаемая {dm_ids[b]}")
        while True:
            response_b = input(f"   {dm_ids[b]} принимает? (y/n): ").strip().lower()
            if response_b in ['y', 'yes', 'да', 'д']:
                accept_b = True
                break
            elif response_b in ['n', 'no', 'нет', 'н']:
                accept_b = False
                break
            else:
                print("Пожалуйста, введите 'y', 'n'")
    else:
        accept_b = False  # Не спрашиваем
    
    return accept_a, accept_b

def print_ici_matrix(self, ici_matrix: np.ndarray, dm_ids: List[str]):
    m = len(dm_ids)
    
    print("      " + "".join([f"{dm_id:>10}" for dm_id in dm_ids]))
    print("      " + "-" * (10 * m))

    for i in range(m):
        row_str = f"{dm_ids[i]:>5} |"
        for j in range(m):
            if i == j:
                row_str += f"{'1.0000':>10}" 
            elif np.isnan(ici_matrix[i, j]):
                row_str += f"{'---':>10}" 
            else:
                row_str += f"{ici_matrix[i, j]:>10.4f}"
        print(row_str)
    
    max_ici = np.nanmax(ici_matrix)
    print(f"   Максимальный: {max_ici:.4f} (наиболее несовместимая пара)")

def print_matrix_comparison(self, current: np.ndarray, proposed: np.ndarray, current_label: str, proposed_label: str):
    n = current.shape[0]
    col_width = 10
    
    # Заголовки
    print(f"   {current_label:<{col_width*n}} | {proposed_label}")
    print("   " + "-" * (col_width*n) + "-+-" + "-" * (col_width*n))
    
    # Вывод матриц построчно
    for i in range(n):
        current_row = ""
        for j in range(n):
            current_row += f"{current[i,j]:{col_width}.3f}"
        proposed_row = ""
        for j in range(n):
            proposed_row += f"{proposed[i,j]:{col_width}.3f}"
        
        print(f"   {current_row} | {proposed_row}")
        print()

def print_simple_matrix(self, matrix: np.ndarray):
    n = matrix.shape[0]
    for i in range(n):
        row = "   "
        for j in range(n):
            row += f"{matrix[i,j]:8.3f}"
        print(row)

def load_pcm_from_json(self, file_path: str) -> Dict[str, Any]:
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    alternatives = data['alternatives']
    dms_data = data['dms']
    parameters = data.get('parameters', {})
    
    pcm_matrices = []
    dm_ids = []
    
    for dm in dms_data:
        dm_ids.append(dm['id'])
        
        if 'pcm' in dm:
            pcm_array = np.array(dm['pcm'])
        elif 'scores' in dm:
            scores = np.array(dm['scores'])
            n = len(scores)
            pcm_array = np.ones((n, n))
            
            for i in range(n):
                for j in range(n):
                    if i != j:
                        pcm_array[i, j] = scores[i] / scores[j] if scores[j] != 0 else 1.0
        else:
            raise ValueError(f"Не найдены данные PCM для эксперта {dm['id']}")
        
        pcm_matrices.append(pcm_array)
    
    return {
        'alternatives': alternatives,
        'pcm_matrices': pcm_matrices,
        'dm_ids': dm_ids,
        'parameters': parameters,
        'n_alternatives': len(alternatives),
        'n_dms': len(dm_ids)
    }

def run_interactive_consensus_from_file(self, file_path: str) -> Dict[str, Any]:
    data = self.load_pcm_from_json(file_path)
    self.current_pcms = np.round(copy.deepcopy(data['pcm_matrices']),3)
    self.dm_ids = data['dm_ids']
    self.alternatives = data['alternatives']
        
    print(f"Загружено {len(self.current_pcms)} матриц PCM от экспертов: {self.dm_ids}")
    print(f"Альтернативы: {self.alternatives}")
    print(f"Порог консенсуса: {self.threshold}")
    print(f"Максимальное число итераций: {self.max_iterations}")
    
    print(f"\nНАЧАЛЬНЫЕ МАТРИЦЫ ЭКСПЕРТОВ:")
    for i, pcm in enumerate(self.current_pcms):
        ci, cr = self.check_consistency(pcm)
        status = "Приемлема" if cr < 0.1 else "Неприемлема"
        print(f"\n{self.dm_ids[i]} (CI={ci:.4f}, CR={cr:.4f} - {status}):")
        self.print_simple_matrix(pcm)
    
    m = len(self.current_pcms)
    D = set((i, j) for i in range(m) for j in range(m) if i != j)
    history = []
    
    for t in range(self.max_iterations):
        ici_matrix = self.calculate_ici_matrix(self.current_pcms, D)
        max_ici = np.nanmax(ici_matrix)
        
        # Проверка условий остановки
        if max_ici <= self.threshold:
            print(f"\nДОСТИГНУТ КОНСЕНСУС! Максимальный ICI = {max_ici:.4f} ≤ {self.threshold:.4f}")
            break
        elif len(D) == 0:
            print(f"\nВСЕ ЭКСПЕРТЫ ОТКАЗАЛИСЬ ОТ ИЗМЕНЕНИЙ")
            break
        
        selected_pair = self.find_max_ici_pair(ici_matrix, D)
        if selected_pair is None:
            break
            
        a, b = selected_pair
        
        accept_a, accept_b = self.enhanced_interactive_acceptance_decision(
            selected_pair, t, self.dm_ids, ici_matrix, D)
        
        alpha_a, alpha_b = self.calculate_alpha(ici_matrix, selected_pair, m)
        
        if accept_a and accept_b:
            A_a_temp = self.current_pcms[a].copy()
            A_b_temp = self.current_pcms[b].copy()
    
            self.current_pcms[a] = self.update_pcm(A_a_temp, A_b_temp, alpha_a)
            self.current_pcms[b] = self.update_pcm(A_b_temp, A_a_temp, alpha_b)

            
        elif accept_a and not accept_b:
            # Только первый эксперт принимает
            A_a_temp = self.current_pcms[a].copy()
            A_b_temp = self.current_pcms[b].copy()
            self.current_pcms[a] = self.update_pcm(A_a_temp, A_b_temp, alpha_a)
            D.discard((b,a)) 
            print(f" Удалена направленная пара: ({self.dm_ids[b]}, {self.dm_ids[a]})")
            
        elif not accept_a and accept_b:
            # Только второй эксперт принимает  
            A_a_temp = self.current_pcms[a].copy()
            A_b_temp = self.current_pcms[b].copy()
            
            self.current_pcms[b] = self.update_pcm(A_b_temp, A_a_temp, alpha_b)
            D.discard((a,b)) 
            print(f"Удалена направленная пара: ({self.dm_ids[a]}, {self.dm_ids[b]})")
            
        else:
            # Оба отказываются
            D.discard((a, b)) 
            D.discard((b, a))
            print(f"Удалены направленные пары: ({self.dm_ids[a]}, {self.dm_ids[b]}) и ({self.dm_ids[b]}, {self.dm_ids[a]})")
    
    final_ici_matrix = self.calculate_ici_matrix(self.current_pcms, D)
    weights = self.calculate_decision_maker_weights(final_ici_matrix, m)
    group_pcm = self.aggregate_group_pcm(self.current_pcms, weights)
    group_priorities = self.calculate_priority_vector(group_pcm)
    
    print(f"\n{'='*80}")
    print(f"ФИНАЛЬНАЯ МАТРИЦА ICI:")
    self.print_ici_matrix(final_ici_matrix, self.dm_ids)
    
    results = {
        'final_pcms': self.current_pcms,
        'final_ici_matrix': final_ici_matrix,
        'group_pcm': group_pcm,
        'group_priorities': group_priorities,
        'weights': weights,
        'iterations_used': t + 1,
        'final_D': D,
        'history': history,
        'consensus_achieved': np.nanmax(final_ici_matrix) <= self.threshold,
        'alternatives': self.alternatives,
        'dm_ids': self.dm_ids,
        'parameters': data['parameters']
    }
    
    return results


# Добавляем методы к классу
PeerToPeerConsensusAHP.enhanced_interactive_acceptance_decision = enhanced_interactive_acceptance_decision
PeerToPeerConsensusAHP.print_ici_matrix = print_ici_matrix
PeerToPeerConsensusAHP.print_matrix_comparison = print_matrix_comparison
PeerToPeerConsensusAHP.print_simple_matrix = print_simple_matrix
PeerToPeerConsensusAHP.load_pcm_from_json = load_pcm_from_json
PeerToPeerConsensusAHP.run_interactive_consensus_from_file = run_interactive_consensus_from_file


In [4]:
def print_final_comparison_table(self, results: Dict[str, Any]):    
    alternatives = results['alternatives']
    dm_ids = results['dm_ids']
    final_pcms = results['final_pcms']
    weights = results['weights']
    group_pcm = results['group_pcm']
    group_priorities = results['group_priorities']
    
    n_alternatives = len(alternatives)
    n_dms = len(dm_ids)
    
    print(f"ГРУППОВАЯ МАТРИЦА ПАРНЫХ СРАВНЕНИЙ:")
    self.print_formatted_matrix(group_pcm, alternatives)
    
    print(f"\nГРУППОВЫЕ ПРИОРИТЕТЫ:")
    for i, (alt, priority) in enumerate(zip(alternatives, group_priorities)):
        print(f"   {i+1:2d}. {alt:<15} : {priority:.4f} ({priority*100:.1f}%)")
    
    ranked_indices = np.argsort(-group_priorities)
    print(f"\nРАНЖИРОВАНИЕ АЛЬТЕРНАТИВ:")
    for rank, idx in enumerate(ranked_indices, 1):
        print(f"   {rank:2d} место: {alternatives[idx]:<15}")

def print_formatted_matrix(self, matrix: np.ndarray, alternatives: List[str]):
    n = len(alternatives)
    header = "         " + "".join([f"{alt:>12}" for alt in alternatives])
    print(header)
    print("         " + "-" * (12 * n))
    
    for i in range(n):
        row_str = f"{alternatives[i]:>8} |"
        for j in range(n):
            row_str += f"{matrix[i,j]:>12.4f}"
        print(row_str)

def print_comprehensive_results(self, results: Dict[str, Any]):
    print(f"{'='*120}")
    
    print(f"\nОСНОВНЫЕ РЕЗУЛЬТАТЫ:")
    print(f"   Консенсус достигнут: {'ДА' if results['consensus_achieved'] else 'НЕТ'}")
    print(f"   Количество итераций: {results['iterations_used']}")
    print(f"   Финальный максимальный ICI: {np.nanmax(results['final_ici_matrix']):.4f}")
    print(f"   Пороговое значение: {self.threshold:.4f}")
    
    print(f"\nВЕСА ЭКСПЕРТОВ:")
    for i, (dm_id, weight) in enumerate(zip(results['dm_ids'], results['weights'])):
        print(f"   {dm_id}: {weight:.4f} ({weight*100:.1f}%)")

    self.print_final_comparison_table(results)

def run_comprehensive_interactive_analysis(self, file_path: str) -> Dict[str, Any]:
    """
    Комплексный интерактивный анализ с полным выводом результатов
    """
    # Запускаем интерактивный режим
    results = self.run_interactive_consensus_from_file(file_path)
    
    # Выводим полный отчет
    self.print_comprehensive_results(results)
    
    return results

# Добавляем методы к классу
PeerToPeerConsensusAHP.print_final_comparison_table = print_final_comparison_table
PeerToPeerConsensusAHP.print_formatted_matrix = print_formatted_matrix
PeerToPeerConsensusAHP.print_comprehensive_results = print_comprehensive_results
PeerToPeerConsensusAHP.run_comprehensive_interactive_analysis = run_comprehensive_interactive_analysis

In [25]:
model = PeerToPeerConsensusAHP(threshold=1.03, max_iterations=10)

try:
    results = model.run_comprehensive_interactive_analysis('input2.json')
    
except Exception as e:
    print(f"Произошла ошибка: {e}")

Загружено 3 матриц PCM от экспертов: ['DM1', 'DM2', 'DM3']
Альтернативы: ['A1', 'A2', 'A3', 'A4', 'A5']
Порог консенсуса: 1.03
Максимальное число итераций: 10

НАЧАЛЬНЫЕ МАТРИЦЫ ЭКСПЕРТОВ:

DM1 (CI=0.0435, CR=0.0392 - Приемлема):
      1.000   3.000   5.000   8.000   6.000
      0.333   1.000   3.000   5.000   4.000
      0.200   0.333   1.000   3.000   2.000
      0.125   0.200   0.333   1.000   0.333
      0.167   0.250   0.500   3.000   1.000

DM2 (CI=0.0959, CR=0.0864 - Приемлема):
      1.000   3.000   7.000   9.000   5.000
      0.333   1.000   3.000   7.000   1.000
      0.143   0.333   1.000   5.000   0.200
      0.111   0.143   0.200   1.000   0.200
      0.200   1.000   5.000   5.000   1.000

DM3 (CI=0.1288, CR=0.1160 - Неприемлема):
      1.000   5.000   7.000   7.000   5.000
      0.200   1.000   5.000   5.000   1.000
      0.143   0.200   1.000   5.000   0.333
      0.143   0.200   0.200   1.000   0.200
      0.200   1.000   3.000   5.000   1.000

ИТЕРАЦИЯ 1

МАТРИЦА ИНДИВ

   DM1 принимает? (y/n):  y



ПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ DM2:
   Текущая DM2                                        | Предлагаемая DM2
   ---------------------------------------------------+---------------------------------------------------
        1.000     3.000     7.000     9.000     5.000 |      1.000     3.000     6.497     8.768     5.206

        0.333     1.000     3.000     7.000     1.000 |      0.333     1.000     3.000     6.497     1.360

        0.143     0.333     1.000     5.000     0.200 |      0.154     0.333     1.000     4.465     0.333

        0.111     0.143     0.200     1.000     0.200 |      0.114     0.154     0.224     1.000     0.224

        0.200     1.000     5.000     5.000     1.000 |      0.192     0.736     3.002     4.465     1.000



   DM2 принимает? (y/n):  n


 Удалена направленная пара: (DM2, DM1)

ИТЕРАЦИЯ 2

МАТРИЦА ИНДИВИДУАЛЬНЫХ ИНДЕКСОВ КОНСЕНСУСА (ICI):
             DM1       DM2       DM3
      ------------------------------
  DM1 |    1.0000    1.1983    1.1393
  DM2 |       ---    1.0000    1.0390
  DM3 |    1.1393    1.0390    1.0000
   Максимальный: 1.1983 (наиболее несовместимая пара)

ПРЕДЛОЖЕНИЕ ПО ПЕРЕСМОТРУ:
   DM1 сохранит 0.7385 своего мнения
   DM2 сохранит 0.7615 своего мнения

ПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ DM1:
   Текущая DM1                                        | Предлагаемая DM1
   ---------------------------------------------------+---------------------------------------------------
        1.000     3.000     5.491     8.267     5.703 |      1.000     3.000     5.851     8.453     5.510

        0.333     1.000     3.000     5.491     2.719 |      0.333     1.000     3.000     5.851     2.093

        0.182     0.333     1.000     3.458     1.053 |      0.171     0.333     1.000     3.808     0.682

        0.121     0

...

 Удалена направленная пара: (DM3, DM1)

ИТЕРАЦИЯ 5

МАТРИЦА ИНДИВИДУАЛЬНЫХ ИНДЕКСОВ КОНСЕНСУСА (ICI):
             DM1       DM2       DM3
      ------------------------------
  DM1 |    1.0000       ---    1.0184
  DM2 |       ---    1.0000    1.0390
  DM3 |       ---    1.0390    1.0000
   Максимальный: 1.0390 (наиболее несовместимая пара)

ПРЕДЛОЖЕНИЕ ПО ПЕРЕСМОТРУ:
   DM2 сохранит 0.5 своего мнения
   DM3 сохранит 0.5 своего мнения

ПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ DM2:
   Текущая DM2                                        | Предлагаемая DM2
   ---------------------------------------------------+---------------------------------------------------
        1.000     3.000     7.000     9.000     5.000 |      1.000     3.873     7.000     7.937     5.000

        0.333     1.000     3.000     7.000     1.000 |      0.258     1.000     3.873     5.916     1.000

        0.143     0.333     1.000     5.000     0.200 |      0.143     0.258     1.000     5.000     0.258

        0.111     0.143  

   DM2 принимает? (y/n):  y



ПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ DM3:
   Текущая DM3                                        | Предлагаемая DM3
   ---------------------------------------------------+---------------------------------------------------
        1.000     5.000     7.000     7.000     5.000 |      1.000     3.873     7.000     7.937     5.000

        0.200     1.000     5.000     5.000     1.000 |      0.258     1.000     3.873     5.916     1.000

        0.143     0.200     1.000     5.000     0.333 |      0.143     0.258     1.000     5.000     0.258

        0.143     0.200     0.200     1.000     0.200 |      0.126     0.169     0.200     1.000     0.200

        0.200     1.000     3.000     5.000     1.000 |      0.200     1.000     3.873     5.000     1.000



   DM3 принимает? (y/n):  y



ДОСТИГНУТ КОНСЕНСУС! Максимальный ICI = 1.0232 ≤ 1.0300

ФИНАЛЬНАЯ МАТРИЦА ICI:
             DM1       DM2       DM3
      ------------------------------
  DM1 |    1.0000       ---    1.0232
  DM2 |       ---    1.0000    0.9999
  DM3 |       ---    0.9999    1.0000
   Максимальный: 1.0232 (наиболее несовместимая пара)

ОСНОВНЫЕ РЕЗУЛЬТАТЫ:
   Консенсус достигнут: ДА
   Количество итераций: 6
   Финальный максимальный ICI: 1.0232
   Пороговое значение: 1.0300

ВЕСА ЭКСПЕРТОВ:
   DM1: 0.0000 (0.0%)
   DM2: 0.5000 (50.0%)
   DM3: 0.5000 (50.0%)
ГРУППОВАЯ МАТРИЦА ПАРНЫХ СРАВНЕНИЙ:
                   A1          A2          A3          A4          A5
         ------------------------------------------------------------
      A1 |      1.0000      3.8730      7.0000      7.9370      5.0000
      A2 |      0.2580      1.0000      3.8730      5.9160      1.0000
      A3 |      0.1430      0.2580      1.0000      5.0000      0.2580
      A4 |      0.1260      0.1690      0.2000      1.0000  

In [21]:
def save_results(results, filepath):
    """Сохраняет только финальные результаты без истории итераций в JSON файл."""
    import numpy as np
    import json
    
    def convert_numpy(obj):
        """Рекурсивно конвертирует numpy типы в Python типы"""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (np.integer, np.int64, np.int32)):
            return int(obj)
        elif isinstance(obj, (np.floating, np.float64, np.float32)):
            return float(obj)
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, list):
            return [convert_numpy(item) for item in obj]
        elif isinstance(obj, dict):
            return {k: convert_numpy(v) for k, v in obj.items()}
        return obj
    
    # Создаем чистый словарь только с ключевыми результатами
    clean_results = {
        'alternatives': results['alternatives'],
        'dm_ids': results['dm_ids'],
        'final_pcms': [pcm.tolist() for pcm in results['final_pcms']],
        'final_ici_matrix': results['final_ici_matrix'].tolist(),
        'group_pcm': results['group_pcm'].tolist(),
        'group_priorities': results['group_priorities'].tolist(),
        'weights': results['weights'].tolist(),
        'consensus_achieved': bool(results['consensus_achieved']),
        'iterations_used': int(results['iterations_used']),
        'parameters': results.get('parameters', {}),
        'max_ici': float(np.nanmax(results['final_ici_matrix']))
    }
    
    # Рекурсивно конвертируем все numpy типы
    clean_results = convert_numpy(clean_results)
    
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(clean_results, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Результаты сохранены в {filepath}")

In [22]:
save_results(results, "final_results2.json")

✅ Результаты сохранены в final_results2.json
